In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm
from torchvision.transforms import AutoAugment, AutoAugmentPolicy 

# --- 1. CONFIGURAZIONE ---
def find_paths():
    base_train = '/kaggle/input/dataset/archive'
    if not os.path.exists(base_train): base_train = '/kaggle/input/dataset'
    
    test_img_path = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if root.endswith("test_data/images"):
            test_img_path = root
            break
    
    if test_img_path is None:
        test_img_path = '/kaggle/input/dataset/archive (1)/dataset challenge 2 dl /test_data/images'
        
    test_mask_path = test_img_path.replace('images', 'masks')
    return {'train_img': os.path.join(base_train, 'images'),
            'train_mask': os.path.join(base_train, 'masks'),
            'labels': os.path.join(base_train, 'train_labels.csv'),
            'test_img': test_img_path, 'test_mask': test_mask_path}

PATHS = find_paths()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PRECOMPUTED_DIR_TRAIN = '/kaggle/working/train_precomputed'
PRECOMPUTED_DIR_TEST = '/kaggle/working/test_precomputed'

IMG_SIZE = 256
BATCH_SIZE = 32
N_FOLDS = 5
EPOCHS = 20
LR = 1e-4 
MIXUP_ALPHA = 0.4

# --- 2. UTILITÀ ---
class ReinhardNormalizer:
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])
    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except: return img_rgb

def get_bbox_from_mask(mask):
    if np.sum(mask) == 0: return None
    rows, cols = np.where(mask > 0)
    return np.min(rows), np.max(rows), np.min(cols), np.max(cols)

class TissueDataset(Dataset):
    def __init__(self, dataframe, precomputed_dir, transform=None, is_test=False):
        self.df = dataframe
        self.precomputed_dir = precomputed_dir 
        self.transform = transform
        self.is_test = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row['filename'] if self.is_test else row['sample_index']
        img_path = os.path.join(self.precomputed_dir, img_name)
        
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (IMG_SIZE, IMG_SIZE)) # Fallback
        
        if self.transform: image = self.transform(image)
        
        if self.is_test: return image, img_name
        else: return image, torch.tensor(row['label_encoded'], dtype=torch.long)

def get_transforms():
    train_trans = transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)), 
        AutoAugment(policy=AutoAugmentPolicy.IMAGENET),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    val_trans = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return train_trans, val_trans

# --- 3. MIXUP & MODEL ---
def mixup_data(x, y, alpha=1.0):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    index = torch.randperm(x.size(0)).to(DEVICE)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

def get_model(num_classes):
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)

# --- 4. PRECOMPUTATION ---
def preprocess_and_save_batch(df, src_img_dir, src_mask_dir, output_dir, is_test=False):
    os.makedirs(output_dir, exist_ok=True)
    reinhard = ReinhardNormalizer()
    valid_files = []
    print(f"Preprocessing {len(df)} images -> {output_dir}")
    
    for _, row in tqdm(df.iterrows(), total=len(df)):
        img_name = row['filename'] if is_test else row['sample_index']
        img_p = os.path.join(src_img_dir, img_name)
        # Fix: gestione nome maschera robusta
        mask_name = img_name.replace("img", "mask") if "img" in img_name else img_name.replace(".png", "_mask.png") # Adatta se necessario
        mask_p = os.path.join(src_mask_dir, mask_name)
        out_p = os.path.join(output_dir, img_name)
        
        if os.path.exists(out_p):
            valid_files.append(img_name)
            continue
            
        try:
            image = cv2.imread(img_p)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE)
            
            if mask is not None:
                bbox = get_bbox_from_mask(mask)
                if bbox:
                    y_min, y_max, x_min, x_max = bbox
                    image = image[y_min:y_max+1, x_min:x_max+1]
            
            image = reinhard(image)
            image_pil = Image.fromarray(image)
            image_resized = image_pil.resize((IMG_SIZE, IMG_SIZE), Image.Resampling.LANCZOS)
            image_resized.save(out_p, "PNG")
            valid_files.append(img_name)
        except: continue
    return valid_files

# --- 5. TRAINING ---
def train_fold(fold, train_idx, val_idx, df, precomputed_dir, num_classes):
    print(f"\n=== FOLD {fold+1}/{N_FOLDS} ===")
    train_df = df.iloc[train_idx].reset_index(drop=True)
    val_df = df.iloc[val_idx].reset_index(drop=True)
    
    # Sampler
    counts = train_df['label_encoded'].value_counts().sort_index().values
    weights = 1. / counts
    s_weights = [weights[l] for l in train_df['label_encoded']]
    sampler = WeightedRandomSampler(s_weights, len(s_weights))
    
    train_loader = DataLoader(TissueDataset(train_df, precomputed_dir, get_transforms()[0]), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    val_loader = DataLoader(TissueDataset(val_df, precomputed_dir, get_transforms()[1]), 
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    model = get_model(num_classes)
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss() # Multi-class Loss
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        t_preds, t_targets = [], []
        
        for imgs, lbls in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            
            imgs, targets_a, targets_b, lam = mixup_data(imgs, lbls, MIXUP_ALPHA)
            out = model(imgs)
            loss = mixup_criterion(criterion, out, targets_a, targets_b, lam)
            
            loss.backward()
            optimizer.step()
            t_loss += loss.item()
            
            t_preds.extend(out.argmax(1).cpu().numpy())
            t_targets.extend(targets_a.cpu().numpy()) # Approx target

        scheduler.step()
        
        model.eval()
        v_loss = 0
        v_preds, v_targets = [], []
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                out = model(imgs)
                v_loss += criterion(out, lbls).item()
                v_preds.extend(out.argmax(1).cpu().numpy())
                v_targets.extend(lbls.cpu().numpy())
        
        t_loss /= len(train_loader)
        v_f1 = f1_score(v_targets, v_preds, average='macro')
        print(f"Ep {epoch+1}: Loss {t_loss:.3f} | Val F1 {v_f1:.3f}")
        
        if v_f1 > best_f1:
            best_f1 = v_f1
            torch.save(model.state_dict(), f"fold_{fold}_best.pth")
            
    print(f"Best F1: {best_f1:.4f}")

# --- 6. SUBMISSION ---
def submit(le):
    print("\n--- Submission ---")
    test_files = sorted([f for f in os.listdir(PATHS['test_img']) if f.endswith('.png')])
    test_df = pd.DataFrame({'filename': test_files})
    
    valid_test = preprocess_and_save_batch(test_df, PATHS['test_img'], PATHS['test_mask'], PRECOMPUTED_DIR_TEST, is_test=True)
    test_df = test_df[test_df['filename'].isin(valid_test)].reset_index(drop=True)
    
    test_ds = TissueDataset(test_df, PRECOMPUTED_DIR_TEST, get_transforms()[1], is_test=True)
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    models_list = []
    for i in range(N_FOLDS):
        path = f"fold_{i}_best.pth"
        if os.path.exists(path):
            m = get_model(len(le.classes_))
            m.load_state_dict(torch.load(path))
            m.eval()
            models_list.append(m)
            
    preds, fnames = [], []
    with torch.no_grad():
        for imgs, names in tqdm(loader):
            imgs = imgs.to(DEVICE)
            inputs = [imgs, torch.flip(imgs, [3]), torch.flip(imgs, [2]), torch.rot90(imgs, 1, [2, 3])]
            probs = torch.zeros((imgs.size(0), len(le.classes_))).to(DEVICE)
            
            for m in models_list:
                for x in inputs:
                    probs += torch.softmax(m(x), dim=1)
            
            probs /= (len(models_list) * 4)
            preds.extend(le.inverse_transform(probs.argmax(1).cpu().numpy()))
            fnames.extend(names)
            
    pd.DataFrame({'sample_index': fnames, 'label': preds}).to_csv('submission.csv', index=False)
    print("Done.")

if __name__ == "__main__":
    # A. Setup Multi-Class
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)
    
    # Label Encoder Reale (4 Classi)
    le = LabelEncoder()
    df['label_encoded'] = le.fit_transform(df['label'])
    
    # Precompute Train
    valid_train = preprocess_and_save_batch(df, PATHS['train_img'], PATHS['train_mask'], PRECOMPUTED_DIR_TRAIN)
    df_final = df[df['sample_index'].isin(valid_train)].reset_index(drop=True)
    
    # B. Training Multi-Class
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for fold, (t_idx, v_idx) in enumerate(skf.split(df_final, df_final['label_encoded'])):
        train_fold(fold, t_idx, v_idx, df_final, PRECOMPUTED_DIR_TRAIN, len(le.classes_))
        
    # C. Submission
    submit(le)

Preprocessing 581 images -> /kaggle/working/train_precomputed


100%|██████████| 581/581 [01:10<00:00,  8.20it/s]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth



=== FOLD 1/5 ===


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s] 


Ep 1: Loss 1.390 | Val F1 0.261


...

Ep 18: Loss 1.249 | Val F1 0.342


Ep 19: Loss 1.231 | Val F1 0.342


Ep 20: Loss 1.234 | Val F1 0.354
Best F1: 0.3536

--- Submission ---
Preprocessing 477 images -> /kaggle/working/test_precomputed


100%|██████████| 15/15 [00:14<00:00,  1.03it/s]

Done.
